In [1]:
# ============================================================================
# Imports & one-time setup
# ============================================================================
import sqlite3
import re
from pathlib import Path

import pandas as pd
import numpy as np

# --- NLP / sentiment -------------------------------------------------------
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from nltk.tokenize import sent_tokenize

import textstat
import pysentiment2 as ps2

# --- Transformers (FinBERT — heavy, gated behind a GPU/opt-in flag) --------
import torch
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification

# --- Viz -------------------------------------------------------------------
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

# One-time NLTK data
nltk.download('vader_lexicon', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('punkt', quiet=True)

# Device for FinBERT
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch device: {DEVICE}')

# Paths
DB_PATH = '../data/market.db'
TRANSCRIPTS_DIR = Path('../data/transcripts')

# Style
sns.set_theme(style='whitegrid', context='notebook')
%matplotlib inline

KeyboardInterrupt: 

In [ ]:
# ============================================================================
# Load transcript metadata + validate files on disk
# ============================================================================
conn = sqlite3.connect(DB_PATH)

# Only successful fetches
tx_df = pd.read_sql(
    "SELECT * FROM transcripts WHERE status = 200 ORDER BY ticker, year, quarter",
    conn, parse_dates=['pub_date', 'scrape_time'])

conn.close()

print(f'Transcripts in DB (status=200): {len(tx_df)}')

# Defensive: verify files actually exist on disk
tx_df['file_exists'] = tx_df['file_path'].apply(lambda p: Path(p).exists())

missing = tx_df[~tx_df['file_exists']]
if len(missing) > 0:
    print(f'WARNING: {len(missing)} DB rows have no local file — dropping:')
    for _, r in missing.iterrows():
        print(f'  {r["ticker"]} {r["quarter"]}{r["year"]}: {r["file_path"]}')

tx_df = tx_df[tx_df['file_exists']].copy()
print(f'Transcripts with verified files: {len(tx_df)}')

if len(tx_df) == 0:
    print('\n⚠️  No transcript files on disk. Run 02_transcripts.ipynb first to scrape them.')
    print('   Continuing with empty results — visualizations will be skipped.')
else:
    print(f'\nTickers: {sorted(tx_df["ticker"].unique())}')
    print(f'Total words: {tx_df["word_count"].sum():,}')
    tx_df[['ticker', 'quarter', 'year', 'word_count']].head(10)

In [ ]:
# ============================================================================
# Text preprocessing
#   - Strip Motley Fool boilerplate headers/footers
#   - Sentence tokenize (for FinBERT chunking)
# ============================================================================

def clean_transcript(raw_text: str) -> str:
    """Strip known Motley Fool boilerplate and normalize whitespace."""
    # Remove MF header lines (common patterns)
    lines = raw_text.split('\n')
    cleaned = []
    skip_header = True
    for line in lines:
        stripped = line.strip()
        # Skip common header/footer patterns
        if any(phrase in stripped.lower() for phrase in [
            'image source: getty images',
            'this article was originally published',
            'the motley fool has a disclosure policy',
            'should you invest',
            'where to invest',
            'before you buy stock in',
            'the motley fool stock advisor',
            'see the 10 stocks',
            'when our analyst team has',
            '©', 'all rights reserved',
        ]):
            continue
        # Skip lines that are just URLs or navigation
        if stripped.startswith('http') and len(stripped.split()) == 1:
            continue
        # Once we hit real content, stop skipping
        if len(stripped) > 30:
            skip_header = False
        if not skip_header:
            cleaned.append(line)

    text = '\n'.join(cleaned)
    # Collapse whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def chunk_text(text: str, max_words: int = 256) -> list[str]:
    """Split text into ~max_words chunks at sentence boundaries."""
    sentences = sent_tokenize(text)
    chunks = []
    current_chunk = []
    current_count = 0
    for sent in sentences:
        wc = len(sent.split())
        if current_count + wc > max_words and current_chunk:
            chunks.append(' '.join(current_chunk))
            current_chunk = [sent]
            current_count = wc
        else:
            current_chunk.append(sent)
            current_count += wc
    if current_chunk:
        chunks.append(' '.join(current_chunk))
    return chunks


# Load all transcript texts (one at a time — don't map over a column)
tx_texts = {}
for _, row in tx_df.iterrows():
    fp = Path(row['file_path'])
    raw = fp.read_text(encoding='utf-8')
    key = (row['ticker'], row['quarter'], row['year'])
    tx_texts[key] = clean_transcript(raw)

print(f'Loaded and cleaned {len(tx_texts)} transcripts')
if tx_texts:
    sample_key = list(tx_texts.keys())[0]
    print(f'\nSample ({sample_key}): first 300 chars:')
    print(tx_texts[sample_key][:300])
    print(f'... ({len(tx_texts[sample_key].split())} words total)')

In [ ]:
# ============================================================================
# Tier 1 & 2: VADER + pysentiment2 LM dictionary (fast)
# ============================================================================

# Initialize analyzers
vader = SentimentIntensityAnalyzer()
lm = ps2.LM()  # Loughran-McDonald financial dictionary

sentiment_rows = []

for (ticker, quarter, year), text in tx_texts.items():
    # --- VADER ---
    vader_scores = vader.polarity_scores(text)

    # --- pysentiment2 LM ---
    tokens = lm.tokenize(text)
    lm_score = lm.get_score(tokens)
    # lm_score has: Positive, Negative, Uncertainty, Litigious, Constraining, StrongModal, WeakModal

    # LM net sentiment: (Positive - Negative) / total (range -1 to +1)
    total_lm = lm_score['Positive'] + lm_score['Negative'] + 1  # +1 avoid div0
    lm_net = (lm_score['Positive'] - lm_score['Negative']) / total_lm

    sentiment_rows.append({
        'ticker': ticker,
        'quarter': quarter,
        'year': year,
        'word_count': len(text.split()),
        # VADER
        'vader_compound': vader_scores['compound'],
        'vader_pos': vader_scores['pos'],
        'vader_neg': vader_scores['neg'],
        'vader_neu': vader_scores['neu'],
        # LM counts
        'lm_positive': lm_score['Positive'],
        'lm_negative': lm_score['Negative'],
        'lm_uncertainty': lm_score['Uncertainty'],
        'lm_litigious': lm_score['Litigious'],
        'lm_constraining': lm_score['Constraining'],
        'lm_strong_modal': lm_score['StrongModal'],
        'lm_weak_modal': lm_score['WeakModal'],
        # LM derived
        'lm_net': lm_net,
        'lm_pos_ratio': lm_score['Positive'] / total_lm,
        'lm_neg_ratio': lm_score['Negative'] / total_lm,
    })

df_sent = pd.DataFrame(sentiment_rows)
print(f'Computed VADER + LM scores for {len(df_sent)} transcripts')

if len(df_sent) > 0:
    print('\n=== Sentiment summary ===')
    display(df_sent[['ticker', 'quarter', 'year', 'vader_compound',
                      'lm_net', 'lm_positive', 'lm_negative']]
            .round(4).head(20))
    print(f'\nVADER compound: mean={df_sent["vader_compound"].mean():.4f}  '
          f'std={df_sent["vader_compound"].std():.4f}')
    print(f'LM net:         mean={df_sent["lm_net"].mean():.4f}  '
          f'std={df_sent["lm_net"].std():.4f}')

In [ ]:
# ============================================================================
# Tier 3: FinBERT sentiment (transformers — optional, skip if no GPU)
# ============================================================================

USE_FINBERT = True  # set to False to skip this cell

if USE_FINBERT and len(tx_texts) > 0:
    print(f'Loading FinBERT on {DEVICE} ...')
    finbert = pipeline(
        'sentiment-analysis',
        model='ProsusAI/finbert',
        tokenizer='ProsusAI/finbert',
        device=0 if DEVICE.type == 'cuda' else -1,
        truncation=True,
        max_length=512,
    )
    print('FinBERT loaded.')

    finbert_results = []
    for (ticker, quarter, year), text in tx_texts.items():
        chunks = chunk_text(text, max_words=256)
        chunk_scores = finbert(chunks, batch_size=8)

        # Aggregate: average the probability assigned to each label
        label_probs = {'positive': 0.0, 'negative': 0.0, 'neutral': 0.0}
        for cs in chunk_scores:
            label_probs[cs['label'].lower()] += cs['score']
        n = len(chunk_scores)
        for k in label_probs:
            label_probs[k] /= n

        # Net: positive - negative
        fb_net = label_probs['positive'] - label_probs['negative']
        fb_label = max(label_probs, key=label_probs.get)

        finbert_results.append({
            'ticker': ticker, 'quarter': quarter, 'year': year,
            'finbert_positive': label_probs['positive'],
            'finbert_negative': label_probs['negative'],
            'finbert_neutral': label_probs['neutral'],
            'finbert_net': fb_net,
            'finbert_label': fb_label,
            'finbert_chunks': n,
        })

    df_fb = pd.DataFrame(finbert_results)
    # Merge FinBERT results into df_sent
    df_sent = df_sent.merge(df_fb, on=['ticker', 'quarter', 'year'], how='left')

    print(f'FinBERT scored {len(df_fb)} transcripts '
          f'(avg {df_fb["finbert_chunks"].mean():.0f} chunks each)')
    print(f'FinBERT net: mean={df_sent["finbert_net"].mean():.4f}  '
          f'std={df_sent["finbert_net"].std():.4f}')
else:
    print('FinBERT skipped (USE_FINBERT=False or no transcripts).')
    # Add placeholder columns so downstream code doesn't break
    for col in ['finbert_positive', 'finbert_negative', 'finbert_neutral',
                'finbert_net', 'finbert_label', 'finbert_chunks']:
        if col not in df_sent.columns:
            df_sent[col] = np.nan

In [ ]:
# ============================================================================
# Readability + linguistic features (textstat)
# ============================================================================

readability_rows = []

for (ticker, quarter, year), text in tx_texts.items():
    sentences = sent_tokenize(text)
    words = text.split()
    n_sentences = len(sentences)
    n_words = len(words)

    # Unique word ratio
    unique_ratio = len(set(w.lower() for w in words)) / max(n_words, 1)

    # Avg sentence length
    avg_sent_len = n_words / max(n_sentences, 1)

    readability_rows.append({
        'ticker': ticker, 'quarter': quarter, 'year': year,
        'n_sentences': n_sentences,
        'n_words': n_words,
        'unique_word_ratio': round(unique_ratio, 4),
        'avg_sentence_length': round(avg_sent_len, 1),
        # textstat readability scores
        'flesch_reading_ease': textstat.flesch_reading_ease(text),
        'flesch_kincaid_grade': textstat.flesch_kincaid_grade(text),
        'gunning_fog': textstat.gunning_fog(text),
        'smog_index': textstat.smog_index(text),
        'automated_readability': textstat.automated_readability_index(text),
        'dale_chall_score': textstat.dale_chall_readability_score(text),
    })

df_read = pd.DataFrame(readability_rows)

# Merge into df_sent
df_sent = df_sent.merge(df_read, on=['ticker', 'quarter', 'year'], how='left')

print(f'Computed readability for {len(df_read)} transcripts')
if len(df_read) > 0:
    print('\n=== Readability summary ===')
    print(f'Flesch Reading Ease:  mean={df_read["flesch_reading_ease"].mean():.1f}  '
          f'(higher = easier)')
    print(f'Flesch-Kincaid Grade: mean={df_read["flesch_kincaid_grade"].mean():.1f}')
    print(f'Gunning Fog:          mean={df_read["gunning_fog"].mean():.1f}')
    print(f'Unique word ratio:    mean={df_read["unique_word_ratio"].mean():.3f}')

In [ ]:
# ============================================================================
# Join sentiment features with returns data
# ============================================================================

# Load returns from SQLite
conn = sqlite3.connect(DB_PATH)
returns_df = pd.read_sql(
    "SELECT * FROM returns", conn, parse_dates=['earnings_date'])
conn.close()

print(f'Returns table: {len(returns_df)} events, {returns_df["ticker"].nunique()} tickers')

# The transcripts table has (ticker, quarter, year) while returns has (ticker, earnings_date).
# We need to map transcript quarter/year to an earnings_date.
# Load earnings data to get the mapping
conn = sqlite3.connect(DB_PATH)
earnings_map = pd.read_sql(
    "SELECT ticker, earnings_date FROM earnings", conn, parse_dates=['earnings_date'])
conn.close()

# For each transcript, find the closest earnings_date (same ticker, date near pub_date)
# The transcripts table has pub_date which is the publication date of the article
# We'll merge on ticker + date proximity

# Strategy: join on ticker, then find the earnings_date closest to pub_date
# (within a reasonable window)
merged_rows = []
for _, tx_row in df_sent.iterrows():
    ticker = tx_row['ticker']
    
    # Get earnings dates for this ticker
    ticker_earnings = earnings_map[earnings_map['ticker'] == ticker].copy()
    if ticker_earnings.empty:
        continue
    
    # We need an earnings_date to match. The transcript quarter/year is fiscal,
    # so we can't directly match. Instead, for each transcript, find the
    # earnings date that has the closest matching quarter from the returns table.
    # The returns table join key is (ticker, earnings_date).
    # For simplicity: match on ticker and find the earnings_date whose calendar
    # quarter most closely aligns with the transcript's fiscal quarter.
    
    # Find returns for this ticker near the transcript's pub_date
    # (earnings call happens within a few days of pub_date for MF articles)
    
    # Simple approach: just pass through the transcript pub_date as a proxy
    # for earnings_date and join on ticker + approximate date
    pass

# Better approach: build a bridge table from transcript (ticker, quarter, year) → earnings_date
# using the fiscal calendar mapping from 02_transcripts.ipynb

# For now, do a simpler merge: for each transcript, find the return row
# whose earnings_date is closest to the transcript's pub_date (within 14 days)

joined = []
unmatched = 0

for _, tx_row in df_sent.iterrows():
    ticker = tx_row['ticker']
    # Get returns for this ticker
    ticker_returns = returns_df[returns_df['ticker'] == ticker].copy()
    if ticker_returns.empty:
        unmatched += 1
        continue
    
    # Find the closest earnings_date to the transcript's pub_date
    # Note: pub_date may not exist in df_sent because we didn't carry it through.
    # We need to go back to tx_df for pub_date.
    matched_row = tx_df[(tx_df['ticker'] == ticker) &
                        (tx_df['quarter'] == tx_row['quarter']) &
                        (tx_df['year'] == tx_row['year'])]
    if matched_row.empty:
        unmatched += 1
        continue
    
    pub_date = matched_row.iloc[0]['pub_date']
    
    # Compute absolute day difference to each earnings_date
    ticker_returns['day_diff'] = abs(
        (ticker_returns['earnings_date'] - pub_date).dt.days)
    closest = ticker_returns.loc[ticker_returns['day_diff'].idxmin()]
    
    # Only join if within 30 days (generous window)
    if closest['day_diff'] > 30:
        unmatched += 1
        continue
    
    # Merge the transcript sentiment + returns row
    row_dict = {k: v for k, v in tx_row.items()}
    row_dict.update({
        'matched_earnings_date': closest['earnings_date'],
        'day_diff': closest['day_diff'],
        'return_1d': closest['return_1d'],
        'return_30d': closest['return_30d'],
        'return_90d': closest['return_90d'],
        'abnormal_1d': closest['abnormal_1d'],
        'abnormal_30d': closest['abnormal_30d'],
        'abnormal_90d': closest['abnormal_90d'],
        'vix_close': closest['vix_close'],
        'is_covid': closest['is_covid'],
    })
    joined.append(row_dict)

if joined:
    df_merged = pd.DataFrame(joined)
    print(f'Matched {len(df_merged)} transcripts to returns data')
    print(f'Unmatched: {unmatched}')
    print(f'Mean day_diff: {df_merged["day_diff"].mean():.1f} days')
else:
    print(f'No matches found ({unmatched} unmatched). Need more data.')
    df_merged = pd.DataFrame()

df_merged.head()

In [ ]:
# ============================================================================
# Persist sentiment features to SQLite
# ============================================================================

if len(df_merged) > 0:
    # Select columns for the features table
    feature_cols = [
        'ticker', 'quarter', 'year',
        # VADER
        'vader_compound', 'vader_pos', 'vader_neg', 'vader_neu',
        # LM
        'lm_positive', 'lm_negative', 'lm_uncertainty', 'lm_litigious',
        'lm_constraining', 'lm_strong_modal', 'lm_weak_modal',
        'lm_net', 'lm_pos_ratio', 'lm_neg_ratio',
        # FinBERT
        'finbert_positive', 'finbert_negative', 'finbert_neutral',
        'finbert_net', 'finbert_label', 'finbert_chunks',
        # Readability
        'n_sentences', 'n_words', 'unique_word_ratio', 'avg_sentence_length',
        'flesch_reading_ease', 'flesch_kincaid_grade', 'gunning_fog',
        'smog_index', 'automated_readability', 'dale_chall_score',
        # Returns (matched)
        'matched_earnings_date',
        'return_1d', 'return_30d', 'return_90d',
        'abnormal_1d', 'abnormal_30d', 'abnormal_90d',
        'vix_close', 'is_covid',
    ]
    # Keep only columns that exist
    keep_cols = [c for c in feature_cols if c in df_merged.columns]
    df_features = df_merged[keep_cols].copy()

    conn = sqlite3.connect(DB_PATH)
    df_features.to_sql('sentiment_features', conn, if_exists='replace', index=False)
    saved = conn.execute("SELECT COUNT(*) FROM sentiment_features").fetchone()[0]
    conn.close()
    print(f'Saved {saved} rows to sentiment_features table')
    print(f'Columns: {df_features.columns.tolist()}')
else:
    print('No data to persist (df_merged is empty).')

In [ ]:
# ============================================================================
# Visualizations
# ============================================================================

if len(df_merged) < 3:
    print(f'⚠️  Only {len(df_merged)} matched transcripts — '
          f'skipping visualizations (need ≥ 3 for meaningful plots).')
    print('Run the full-scale fetch in 02_transcripts.ipynb first.')
else:
    # ---- 1. Sentiment correlation matrix ----
    fig, ax = plt.subplots(figsize=(10, 8))
    corr_cols = [c for c in ['vader_compound', 'lm_net', 'finbert_net',
                              'abnormal_1d', 'abnormal_30d', 'abnormal_90d',
                              'vix_close', 'flesch_reading_ease',
                              'unique_word_ratio']
                 if c in df_merged.columns and df_merged[c].notna().any()]
    corr = df_merged[corr_cols].corr()
    sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r',
                center=0, vmin=-1, vmax=1, ax=ax,
                cbar_kws={'label': 'Pearson r'})
    ax.set_title('Sentiment × Returns correlation matrix', fontsize=14, pad=15)
    plt.tight_layout()
    plt.show()

    # ---- 2. Scatter: VADER compound vs abnormal_1d ----
    if 'vader_compound' in df_merged.columns and 'abnormal_1d' in df_merged.columns:
        fig, ax = plt.subplots(figsize=(8, 6))
        colors = df_merged['is_covid'].map({True: '#e74c3c', False: '#3498db'})
        ax.scatter(df_merged['vader_compound'], df_merged['abnormal_1d'],
                   c=colors, alpha=0.7, edgecolors='white', s=80)
        for _, row in df_merged.iterrows():
            ax.annotate(f"{row['ticker']} {row['quarter']}{row['year']}",
                        (row['vader_compound'], row['abnormal_1d']),
                        fontsize=7, alpha=0.7,
                        textcoords='offset points', xytext=(5, 5))
        ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
        ax.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
        ax.set_xlabel('VADER Compound Sentiment')
        ax.set_ylabel('Abnormal 1-Day Return (XLK-adjusted)')
        ax.set_title('Earnings Call Sentiment vs Next-Day Abnormal Return')
        # Legend
        from matplotlib.patches import Patch
        legend_elements = [Patch(facecolor='#3498db', label='Normal'),
                           Patch(facecolor='#e74c3c', label='COVID window')]
        ax.legend(handles=legend_elements, loc='best')
        plt.tight_layout()
        plt.show()

    # ---- 3. By ticker: mean sentiment ----
    if 'vader_compound' in df_merged.columns:
        ticker_sent = df_merged.groupby('ticker').agg(
            vader_mean=('vader_compound', 'mean'),
            vader_std=('vader_compound', 'std'),
            count=('vader_compound', 'count'),
            abn_1d_mean=('abnormal_1d', 'mean'),
        ).sort_values('vader_mean')

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

        # Sentiment by ticker
        bars = ax1.barh(ticker_sent.index, ticker_sent['vader_mean'],
                        xerr=ticker_sent['vader_std'], capsize=3,
                        color='steelblue', alpha=0.8)
        ax1.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
        ax1.set_xlabel('Mean VADER Compound')
        ax1.set_title('Sentiment by Ticker')

        # Abnormal return by ticker
        ax2.barh(ticker_sent.index, ticker_sent['abn_1d_mean'],
                 color='coral', alpha=0.8)
        ax2.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
        ax2.set_xlabel('Mean Abnormal 1d Return')
        ax2.set_title('Abnormal 1d Return by Ticker')

        plt.tight_layout()
        plt.show()

    # ---- 4. Sentiment vs readability ----
    if 'vader_compound' in df_merged.columns and 'flesch_reading_ease' in df_merged.columns:
        fig, ax = plt.subplots(figsize=(8, 6))
        ax.scatter(df_merged['flesch_reading_ease'], df_merged['vader_compound'],
                   c='steelblue', alpha=0.7, s=80, edgecolors='white')
        ax.set_xlabel('Flesch Reading Ease (higher = easier)')
        ax.set_ylabel('VADER Compound Sentiment')
        ax.set_title('Sentiment vs Readability')
        plt.tight_layout()
        plt.show()

## Summary & next steps

This notebook computed:
- **VADER** compound sentiment scores
- **Loughran-McDonald** financial dictionary counts (positive, negative,
  uncertainty, litigious, modal words)
- **FinBERT** transformer-based sentiment (if enabled and GPU available)
- **Readability** metrics (Flesch-Kincaid, Gunning Fog, etc.)
- **Linguistic** features (unique word ratio, avg sentence length)

Results were matched to XLK-adjusted abnormal returns and saved to the
`sentiment_features` table in `market.db`.

**If you have < 30 matched transcripts:** the correlation analysis is
underpowered. Run `02_transcripts.ipynb` cell 8 to fetch all ~360 transcripts
first, then re-run this notebook.

**Next notebook ideas:**
- `04_modeling.ipynb` — LightGBM regression predicting abnormal_30d from
  sentiment + VIX + readability features, with SHAP explainability
- `05_portfolio.ipynb` — backtest a long/short strategy based on sentiment
  surprise (actual sentiment vs expected by VIX regime)